# Basic homework rerun: LoRA rank 4 and rank 16 with logged evidence (background run)

The course notebook `lora_format_tuning.ipynb`, unchanged except: it runs **r = 4 and then r = 16** inside one background script (each rank in a
fresh process with a fresh model, so no Restart session is needed), logs the trainable-parameter count and adapter shape (`rank_check_r<r>.json`),
uses Drive (`MyDrive/aie2026/lec02_v3/basic/`), and turns progress bars off. Section 8 (4-bit loading) is not part of this rerun.

| Cell | What | Time |
| --- | --- | --- |
| 1 GPU check, 2 install | course cells | ~1–2 min |
| 3 SETUP | mounts Google Drive | <1 min |
| 4 SCRIPT | writes `/content/lec02_v3/run_basic.py` | instant |
| 5 LAUNCH | starts it with `nohup … &`, returns after 20 s; refuses to start a second copy | 20 s |
| 6 POLL | status + last 20 log lines | instant |
| 7 FINISH | rank comparison table, zip | <1 min |

Expected background run time on a T4 ≈ 15–20 minutes (≈ 7–9 min per rank). Outputs: `result_lora_r4.json`, `result_lora_r16.json`,
`rank_check_r4.json`, `rank_check_r16.json`, `eval_{before,after}_r{4,16}.json`, `rank_comparison.json`, `log.txt`.

## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [ ]:
# [Cell 1] GPU check
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

In [ ]:
# [Cell 2] install
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## Setup and launch

Run cells 3, 4 and 5 once. Cell 5 returns after about 20 seconds; both ranks then run in the background.

In [ ]:
# [Cell 3] SETUP: mount Google Drive and set paths (run again after any reconnect)
import os, sys, json, time, shlex, shutil, subprocess
from pathlib import Path
RUN_NAME = "basic"
USE_DRIVE = True   # strongly recommended: results and the log survive a disconnect
LOCAL = Path("/content/lec02_v3"); LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = Path("/content/drive/MyDrive/aie2026/lec02_v3")
if USE_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive")   # school account: allow the pop-up, pick the school Google account, allow all permissions
        except Exception as e:
            print(f"Drive mount failed ({type(e).__name__}: {e}).\n"
                  "Try: allow pop-ups / third-party cookies for colab.research.google.com, pick the SAME school account that owns "
                  "this notebook, tick every permission, then rerun this cell. If it still fails: Runtime -> Disconnect and delete "
                  "runtime, reconnect (T4) and rerun. As a last resort set USE_DRIVE = False (results then live only in /content).")
            raise
    OUT = DRIVE_ROOT / RUN_NAME
else:
    OUT = LOCAL / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
RUN_PY = LOCAL / "run_basic.py"
LOG = OUT / "log.txt"
LAUNCH_WAIT_S = 20

def run_state():
    """Lock contents if the detached run is alive on this VM, else None"""
    lock = OUT / "run.lock"
    try: d = json.loads(lock.read_text())
    except (OSError, ValueError): return None
    if d.get("boot_id") != Path("/proc/sys/kernel/random/boot_id").read_text().strip(): return None
    try: os.kill(int(d["pid"]), 0)
    except (OSError, ValueError, KeyError): return None
    return d

def tail(n=20):
    if not LOG.exists(): return "(no log yet)"
    return "\n".join(LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])

print("Outputs ->", OUT, "| script ->", RUN_PY, "| running now:", bool(run_state()))

In [ ]:
%%writefile /content/lec02_v3/run_basic.py
# [Cell 4] SCRIPT: written to disk, not run here; the LAUNCH cell runs it in the background
"""Lecture 02 Basic: course notebook run for r = 4 and r = 16 in one background job. Written by the notebook.
Usage: python -u run_basic.py [--stage all|rank --rank R]"""

# ---------------------------------------------------------------- runtime: log, lock, child stages
import os, sys, json, time, socket, signal, subprocess, argparse, contextlib
from pathlib import Path
for _k in ("TQDM_DISABLE", "HF_HUB_DISABLE_PROGRESS_BARS", "HF_DATASETS_DISABLE_PROGRESS_BARS"):
    os.environ.setdefault(_k, "1")            # no progress bars: plain log lines only
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
DRY_RUN = os.environ.get("LEC02_DRY_RUN") == "1"   # CPU test mode used only by validate.py: stub model, no training

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def display(x):  # notebook display() -> plain text in the log
    print(x.to_string() if hasattr(x, "to_string") else x, flush=True)

def boot_id():
    try: return Path("/proc/sys/kernel/random/boot_id").read_text().strip()
    except OSError: return socket.gethostname()

def lock_owner(lock):
    """Lock contents if the process that wrote it is still alive on this VM, else None"""
    try: d = json.loads(lock.read_text())
    except (OSError, ValueError): return None
    if d.get("boot_id") != boot_id(): return None
    try: os.kill(int(d["pid"]), 0)
    except (OSError, ValueError, KeyError): return None
    try:
        if d.get("script") and d["script"] not in Path(f"/proc/{d['pid']}/cmdline").read_bytes().decode(errors="replace"):
            return None                       # pid reused by another program
    except OSError: pass
    return d

def acquire_lock(lock):
    other = lock_owner(lock)
    if other and int(other["pid"]) != os.getpid():
        log(f"another run is active (pid {other['pid']}, started {other.get('started')}); this copy exits")
        sys.exit(3)
    lock.write_text(json.dumps({"pid": os.getpid(), "boot_id": boot_id(), "script": Path(__file__).name,
                                "started": time.strftime("%Y-%m-%d %H:%M:%S")}))
    time.sleep(1)
    if json.loads(lock.read_text()).get("pid") != os.getpid():   # lost a simultaneous start
        log("another run took the lock; this copy exits"); sys.exit(3)
    def _stop(signum, frame):
        log(f"received signal {signum}; stopping"); release_lock(lock); sys.exit(128 + signum)
    signal.signal(signal.SIGTERM, _stop)

def release_lock(lock):
    try:
        if json.loads(lock.read_text()).get("pid") == os.getpid(): lock.unlink()
    except (OSError, ValueError): pass

def run_stage(args, retries=1):
    """Run one stage of this script in a fresh child process (fresh GPU memory). Output goes to the same log"""
    for attempt in range(1, retries + 2):
        log(f"==> stage {' '.join(args)} (attempt {attempt})")
        rc = subprocess.call([sys.executable, "-u", __file__, *args])
        log(f"<== stage {' '.join(args)} exit code {rc}")
        if rc == 0: return True
    return False

def require_gpu():
    if DRY_RUN: return
    import torch
    if not torch.cuda.is_available():
        log("ERROR: no GPU. Runtime -> Change runtime type -> T4 GPU, then run the SETUP and LAUNCH cells again"); sys.exit(2)
    log("GPU:", torch.cuda.get_device_name(0))

def heartbeat_callback():
    """One log line every 5 optimizer steps (training loss / validation loss lines come from the Trainer itself)"""
    from transformers import TrainerCallback
    class Heartbeat(TrainerCallback):
        def on_train_begin(self, args, state, control, **kw):
            self.t0 = time.time(); log(f"training starts: {state.max_steps} steps")
        def on_step_end(self, args, state, control, **kw):
            if state.global_step % 5 == 0 or state.global_step == state.max_steps:
                log(f"step {state.global_step}/{state.max_steps} | epoch {state.epoch:.2f} | {time.time() - self.t0:.0f} s")
    return Heartbeat()

# ---------------------------------------------------------------- Basic: settings for the background run
OUT = Path(os.environ.get("LEC02_OUT") or "/content/drive/MyDrive/aie2026/lec02_v3/basic")
if not OUT.parent.exists():
    sys.exit(f"Output folder {OUT.parent} does not exist: run the SETUP cell (it mounts Google Drive) first")
OUT.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT = Path(os.environ.get("LEC02_WORK", "/content/qlora_lab"))   # checkpoints stay on the VM disk
RANKS = [4, 16]

def compare_ranks():
    """rank_comparison.json: settings evidence and whether the after-training outputs differ between ranks"""
    res = {r: json.loads((OUT / f"result_lora_r{r}.json").read_text(encoding="utf-8")) for r in RANKS}
    after = {r: json.loads((OUT / f"eval_after_r{r}.json").read_text(encoding="utf-8"))["rows"] for r in RANKS}
    a, b = RANKS
    same = [x["question"] for x, y in zip(after[a], after[b]) if x["raw"] == y["raw"]]
    comp = {"ranks": {r: {"rank_check": {k: v for k, v in res[r]["rank_check"].items() if k != "loss_log"},
                          "before": res[r]["before"], "after": res[r]["after"],
                          "knowledge_after": [(k["Question"], k["After-training classification"]) for k in res[r]["knowledge"]]}
                      for r in RANKS},
            "after_outputs_identical": f"{len(same)}/{len(after[a])}",
            "after_outputs_differing": [x["question"] for x, y in zip(after[a], after[b]) if x["raw"] != y["raw"]]}
    save_json(OUT / "rank_comparison.json", comp)
    log("rank comparison:", json.dumps({r: comp["ranks"][r]["rank_check"].get("trainable_params") for r in RANKS}),
        "| identical after-training outputs:", comp["after_outputs_identical"])

def save_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def orchestrate():
    lock = OUT / "run.lock"
    acquire_lock(lock)
    try:
        (OUT / "ALL_DONE.json").unlink(missing_ok=True)
        log(f"basic rank check | ranks {RANKS} | out {OUT} | pid {os.getpid()}{' | DRY RUN' if DRY_RUN else ''}")
        require_gpu()
        failed = []
        for r in RANKS:
            if (OUT / f"r{r}" / "done.json").exists():
                log(f"r={r}: already done, skipped"); continue
            if not run_stage(["--stage", "rank", "--rank", str(r)]): failed.append(r)
        status = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "ranks_failed": failed}
        save_json(OUT / "STATUS.json", status)
        if not failed:
            compare_ranks(); save_json(OUT / "ALL_DONE.json", status); log("ALL DONE")
        else:
            log("failed ranks:", failed, "- rerun LAUNCH to retry them")
        return 0 if not failed else 1
    finally:
        release_lock(lock)

ap = argparse.ArgumentParser()
ap.add_argument("--stage", choices=["all", "rank"], default="all")
ap.add_argument("--rank", type=int)
ARGS = ap.parse_args()
if ARGS.stage == "all":
    sys.exit(orchestrate())

# ================================================================ one rank = the course notebook, top to bottom
LORA_R = ARGS.rank
log(f"===== r = {LORA_R} =====")

# ---- 1. settings (course cell; lora_r = LORA_R)
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "seed": 42,
    "num_train_epochs": 20,  # Number of passes over training data (14 examples). 2 steps per pass gives 40 steps (4–5 minutes on a T4 in the original Japanese run)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": LORA_R,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/qlora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Save location when USE_DRIVE=True
}
CONFIG["work_dir"] = str(OUT / f"r{LORA_R}")   # background run: each rank writes to its own Drive folder
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # background run: WORK is already on Drive (mounted by the SETUP cell)
_unused = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

# ---- 2. FAQ and training data (course cell, unchanged)
FAQ = {
    "deadline": "The assignment deadline is September 30 at 23:59 (Japan time).",
    "destination": "Submit through the 'Assignments' section of the learning portal.",
    "format": "Submit one PDF file named student_id_assignment_number.pdf.",
    "office_hours": "Questions are welcome during office hours every Tuesday from 17:00 to 18:00.",
    "late": "Late submissions are accepted within 48 hours of the deadline, with a grade penalty.",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# With evidence: (FAQ key, question templates, required keywords in the expected answer)
GROUNDED = {
    "deadline": (["When is the assignment deadline?", "By when should I submit?", "Tell me the deadline", "What is the submission deadline?"], ["September 30", "23:59"]),
    "destination": (["Where should I submit?", "Where do I upload it?", "Where can I submit the assignment?", "Where should I submit my work?"], ["learning portal", "Assignments"]),
    "format": (["Tell me the submission format", "Are there any file naming rules?", "Which file format should I submit?", "Can I submit a format other than PDF?"], ["PDF"]),
    "office_hours": (["When can I ask questions?", "When are office hours?", "Is there a time to ask the instructor questions?", "What times are available for consultation?"], ["Tuesday", "17:00"]),
    "late": (["What happens if I submit late?", "Are submissions accepted after the deadline?", "What are the late submission rules?", "Tell me how submissions after the deadline are handled"], ["48 hours", "penalty"]),
}
# Without evidence: absent from the FAQ, so needs_review=true is correct
UNGROUNDED = [
    "Can I resubmit?", "Tell me how missed lectures are handled", "Can I see other participants' scores?",
    "When will grades be released?", "Can the submission deadline be extended?", "Are group submissions allowed?",
    "How are points allocated for assignments?", "Will lecture recordings be made available?",
]

# Additional test-only questions (excluded from train / valid)
TEST_EXTRA_GROUNDED = {
    "deadline": ["Tell me the submission deadline date and time", "On what date and at what time is the assignment due?"],
    "destination": ["Where do I hand in the assignment?", "Tell me where to send my submission"],
    "format": ["What format should the submitted file use?", "What file format and how many files should I submit?"],
    "office_hours": ["Tell me the day and time of office hours", "On which day and from what time can I ask questions?"],
    "late": ["What happens if I miss the submission deadline?", "Will an assignment submitted after the deadline be accepted?"],
}
TEST_EXTRA_UNGROUNDED = [
    "Is there a character count limit for assignments?", "Will I receive feedback on my submitted assignment?",
    "Is a particular citation style required?", "Do office hours require an appointment?",
]
UNGROUNDED_ANSWER = "This FAQ does not contain that information. Confirmation from the course organizers is needed."

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # Fix the first 2 templates to train, the third to valid, and the fourth to test (keep paraphrases separate)
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # Assign each group of 4 questions without evidence as train 2 / valid 1 / test 1
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}

# train has 14 examples (10 with evidence, 4 without). Set repetitions with num_train_epochs in section 4 instead of duplicating data
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "Questions are duplicated across train / valid / test. Review the templates"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("One training example | Question:", train[0]["question"], "| Desired output:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- 3. load the model (course cell, unchanged)
if DRY_RUN:
    tok, DEVICE, BASE_MEMORY_GB, model = None, "cpu", 0.0, "stub-model"
    SYSTEM = "stub"
    def build_messages(example):
        return [{"role": "user", "content": example["question"]}]
    def generate(model, example):
        return json.dumps({"answer": UNGROUNDED_ANSWER if example["needs_review"] else "The assignment deadline is September 30 at 23:59 (Japan time).",
                           "needs_review": example["needs_review"]})
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    # No quantization. Keep weights in 32-bit (specify dtype because omitting it loads the distributed precision)
    model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
    model.config.use_cache = True
    BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
    print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| Weight memory (GB):", BASE_MEMORY_GB)

    # The prompt instructs the format. Before training, check whether the model follows it
    SYSTEM = "You answer course FAQ questions. Use only the provided FAQ as evidence and always answer in one JSON line. Format: {\"answer\": string, \"needs_review\": boolean}. For information absent from the FAQ, write 'Confirmation is needed' in answer and set needs_review to true."

    def build_messages(example):
        user = f"FAQ:\n{example['faq']}\n\nQuestion: {example['question']}"
        return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

    def generate(model, example):
        # enable_thinking=False: leave the thinking block empty and generate only the answer (explicit for consistent conditions across models)
        prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---- 3. evaluate before training (course cell, unchanged)
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

def check_format(text: str):
    '''Check JSON parsing, required keys, and types. Also return the reason for failure'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Content passes if needs_review matches the expected answer and answer contains required keywords'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "Pass" if ok else "Fail"

def evaluate(model, label: str):
    '''Generate and score the 21 test questions, returning a summary and per-question results. Save original outputs in eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # If the format is broken but a JSON-like portion can be parsed, score content separately
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# Per-question outputs. Compare with expected needs_review and check for missing keys and agreement with FAQ wording
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "Before-training output": r["raw"],
                       "Format": mark(r["format_ok"]), "Format failure reason": r["format_reason"], "Content": mark(r["content_ok"])} for r in before_rows]))

# ---- 4. LoRA training (course cell; no progress bars, heartbeat log, local checkpoints) + rank check
if DRY_RUN:
    class _StubTuned:
        def disable_adapter(self): return contextlib.nullcontext()
    tuned = _StubTuned()
    RANK_CHECK = {"CONFIG_lora_r": CONFIG["lora_r"], "peft_config_r": CONFIG["lora_r"], "dry_run": True, "trainable_params": 0, "loss_log": []}
    (WORK / f"rank_check_r{CONFIG['lora_r']}.json").write_text(json.dumps(RANK_CHECK, indent=2), encoding="utf-8")
else:
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    def to_chat(example):
        # Training data pairs inputs (system + user) with desired outputs (assistant). Use the same build_messages as for evaluation
        return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

    train_ds = Dataset.from_list([to_chat(e) for e in train])
    valid_ds = Dataset.from_list([to_chat(e) for e in valid])

    model.config.use_cache = False
    # Qwen3.5 uses linear attention in 18 of 24 layers; only the remaining 6 have q/k/v/o_proj. All layers have gate/up/down_proj (MLP)
    lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    USE_AMP = False  # Normally leave unchanged. True enables 16-bit mixed-precision computation (CUDA only; weights stay 32-bit). Faster, but loss may become nan
    sft = SFTConfig(output_dir=str(LOCAL_CKPT / f"ckpt_r{CONFIG['lora_r']}"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                    learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                    fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", disable_tqdm=True, seed=CONFIG["seed"])
    trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok, callbacks=[heartbeat_callback()])
    trainer.model.print_trainable_parameters()   # Fraction of parameters trained with LoRA
    t0 = time.time()
    trainer.train()  # See 4b to resume if interrupted
    print(f"Training time: {time.time()-t0:.0f} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f}")
    ADAPTER_DIR = WORK / "adapter"
    trainer.model.save_pretrained(ADAPTER_DIR)
    if USE_DRIVE:
        import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
    tuned = trainer.model
    tuned.config.use_cache = True
    _ = tuned.eval()  # Training mode enables dropout and makes evaluation outputs vary (assignment suppresses displaying the whole model)

    # rank check: evidence that the attached adapter really has rank CONFIG["lora_r"]
    cfg = trainer.model.peft_config["default"]
    trainable, total = trainer.model.get_nb_trainable_parameters()
    lora_A = next((n, tuple(p.shape)) for n, p in trainer.model.named_parameters() if "lora_A" in n)
    RANK_CHECK = {"CONFIG_lora_r": CONFIG["lora_r"], "peft_config_r": cfg.r, "peft_config_alpha": cfg.lora_alpha,
                  "trainable_params": trainable, "total_params": total, "trainable_pct": round(100 * trainable / total, 4),
                  "example_lora_A": {"name": lora_A[0], "shape": lora_A[1]},
                  "train_runtime_s": next((e["train_runtime"] for e in trainer.state.log_history if "train_runtime" in e), None),
                  "loss_log": [[e["step"], e["loss"]] for e in trainer.state.log_history if "loss" in e]}
    assert cfg.r == CONFIG["lora_r"] == lora_A[1][0], "The attached adapter does not have the configured rank"
    (WORK / f"rank_check_r{CONFIG['lora_r']}.json").write_text(json.dumps(RANK_CHECK, indent=2), encoding="utf-8")
    log("rank check:", json.dumps({k: v for k, v in RANK_CHECK.items() if k != "loss_log"}))

# ---- 5. evaluate after training (course cell, unchanged)
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Questions that passed both format and content before training but failed afterward (regressions)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Regressions:", regressed)

# Original per-question outputs. Visually check format (JSON, required keys, types) and content (matching needs_review and required keywords) judgments
detail = pd.DataFrame([{"Question": b["question"], "Expected needs_review": b["gold_needs_review"],
                        "Before-training output": b["raw"], "Before format/content": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "After-training output": a["raw"], "After format/content": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

# ---- 6. knowledge questions (course cell, unchanged)
KNOWLEDGE_QUESTIONS = [
    "When is the final assignment deadline for this course?",       # Absent from FAQ (fictional)
    "What is the instructor's name?",                     # Absent from FAQ
    "What is the topic of the next lecture?",                 # Absent from FAQ
    "When is the assignment deadline?",                   # In FAQ (control)
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA is inserted into base-model layers, so generation with `model` also uses adapters after training. Temporarily disable adapters for before-training outputs
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Question": q, "In FAQ": q in grounded_questions, "Before-training classification": classify(before_raw), "Before-training output": before_raw,
                 "After-training classification": classify(after_raw), "After-training output": after_raw})
display(pd.DataFrame(knowledge_rows))

# ---- 7. result file (course cell; file named by rank, rank_check added)
import importlib.metadata as _md
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {p: _md.version(p) for p in ["transformers", "peft", "trl", "torch"]} if not DRY_RUN else {"dry_run": True},
    "method": "LoRA (no quantization)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,
    "rank_check": RANK_CHECK,   # Set False if using the instructor's results
}
(WORK / f"result_lora_r{CONFIG['lora_r']}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in [f"result_lora_r{CONFIG['lora_r']}.json", f"rank_check_r{CONFIG['lora_r']}.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Saved to Drive:", CONFIG["drive_dir"])

# ---------------------------------------------------------------- copy this rank's files to the run folder, mark done (last)
import shutil
r = CONFIG["lora_r"]
for src_name, dst_name in [(f"result_lora_r{r}.json", f"result_lora_r{r}.json"), (f"rank_check_r{r}.json", f"rank_check_r{r}.json"),
                           ("eval_before.json", f"eval_before_r{r}.json"), ("eval_after.json", f"eval_after_r{r}.json")]:
    shutil.copy(WORK / src_name, OUT / dst_name)
(WORK / "done.json").write_text(json.dumps({"rank": r, "finished": time.strftime("%Y-%m-%d %H:%M:%S")}), encoding="utf-8")
log(f"r = {r} done")

In [ ]:
# [Cell 5] LAUNCH: start the run in the background and return at once (safe to rerun: never starts a second copy)
if (OUT / "ALL_DONE.json").exists():
    print("Already finished. Go to the FINISH cell.")
elif run_state():
    print(f"Already running (pid {run_state()['pid']}, started {run_state()['started']}). Use the POLL cell; do not launch again.")
else:
    shutil.copy(RUN_PY, OUT / RUN_PY.name)   # keep the exact script next to the results
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(f"\n===== launch {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    cmd = f"nohup {shlex.quote(sys.executable)} -u {shlex.quote(str(RUN_PY))} --stage all >> {shlex.quote(str(LOG))} 2>&1 &"
    subprocess.Popen(cmd, shell=True, env={**os.environ, "LEC02_OUT": str(OUT)}, start_new_session=True,
                     stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(LAUNCH_WAIT_S)
    print("running:", bool(run_state()), "| log:", LOG)
    print(tail(10))

## Poll

Rerun every 5 minutes. Expected order: r=4 before-eval, training (step lines), after-eval, knowledge; then the same for r=16.

In [ ]:
# [Cell 6] POLL: rerun whenever you like; prints status and the last 20 log lines, then exits
state = run_state()
age = time.time() - LOG.stat().st_mtime if LOG.exists() else None
age_txt = f"last log write {age:.0f} s ago" if age is not None else "no log yet"
print(f"{time.strftime('%H:%M:%S')} | running: {bool(state)}{' (pid ' + str(state['pid']) + ')' if state else ''}"
      f" | finished: {(OUT / 'ALL_DONE.json').exists()} | {age_txt}")
print("done markers:", sorted(str(p.parent.relative_to(OUT)) for p in OUT.glob("*/done.json")))
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip())
except Exception:
    pass
print("-" * 80); print(tail(20))
if not state and not (OUT / "ALL_DONE.json").exists() and LOG.exists():
    print("\nNOT running and not finished: rerun the LAUNCH cell to resume (finished parts are skipped).")

## Finish

In [ ]:
# [Cell 7] FINISH: check completion, rebuild the summary / plots, zip the results
WAIT_MINUTES = 0   # >0 waits here (one short line per minute) until the run finishes; 0 = just check now
t_end = time.time() + 60 * WAIT_MINUTES
while not (OUT / "ALL_DONE.json").exists() and run_state() and time.time() < t_end:
    time.sleep(60); print(time.strftime("%H:%M:%S"), "still running |", tail(1))
if not (OUT / "ALL_DONE.json").exists():
    print("Not finished yet:", "still running - use POLL and come back" if run_state() else
          "stopped early - rerun LAUNCH to resume (see STATUS.json / log.txt)")
else:
    comp = json.loads((OUT / "rank_comparison.json").read_text(encoding="utf-8"))
    print(json.dumps(comp, indent=1, ensure_ascii=False)[:4000])
    archive = shutil.make_archive(str(LOCAL / "lec02_v3_basic"), "zip", root_dir=OUT)
    if Path(archive).resolve() != (OUT.parent / Path(archive).name).resolve():
        shutil.copy(archive, OUT.parent / Path(archive).name)
    print("zip:", archive, "| copy on Drive:", OUT.parent / Path(archive).name)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as e:
        print(f"Automatic download unavailable ({type(e).__name__}); download the zip from the Files panel or Drive")